In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv(r"D:\MASTERS\FOML\assignment\Project\Project@\ML-Health-Insurance\Dataset of health insurance portfolio.xlsx - Sheet1 (1).csv")

df.head()

,ID,ID_policy,ID_insured,period,date_effect_insured,date_lapse_insured,date_effect_policy,date_lapse_policy,year_effect_insured,year_lapse_insured,...,C_H,C_GI,C_II,C_IE_P,C_IE_S,C_IE_T,C_GE_P,C_GE_S,C_GE_T,C_C
0,1_1,1,1,2017,6/15/1993,12/20/2017,6/15/1993,NaN,1993,2017.0,...,H1,99.0,85.0,18.0,52.0,59.0,4.0,14.0,96.0,C6
1,1_2,1,2,2017,6/15/1993,NaN,6/15/1993,NaN,1993,NaN,...,H1,99.0,85.0,18.0,52.0,59.0,4.0,14.0,96.0,C6
2,2_1,2,1,2017,6/16/1993,NaN,6/16/1993,NaN,1993,NaN,...,H6,100.0,94.0,26.0,11.0,85.0,7.0,1.0,100.0,C6
3,3_3,3,3,2017,6/15/1993,NaN,6/15/1993,NaN,1993,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3_5,3,5,2017,6/15/1993,NaN,6/15/1993,NaN,1993,NaN,...,H6,99.0,81.0,32.0,34.0,69.0,9.0,4.0,98.0,C6


In [4]:
df["period"].value_counts().sort_index()

period
2017    78459
2018    73970
2019    76282
Name: count, dtype: int64

In [5]:
#columns required for renewal filtering and lapse date auditing
required_columns = [
    "ID", "ID_policy", "ID_insured",
    "period", "lapse", "new_business", "premium",
    "date_effect_insured", "date_effect_policy",
    "date_lapse_insured", "date_lapse_policy",
    "year_effect_insured", "year_effect_policy",
    "year_lapse_insured", "year_lapse_policy"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

print("Missing required columns:", missing_columns)

Missing required columns: []


In [6]:
#Creating a seperate copy for dataset preperation
df_original = df.copy()

df_prepared = df.copy()

print("Original shape:", df_original.shape)
print("Working shape:", df_prepared.shape)

Original shape: (228711, 42)
Working shape: (228711, 42)


In [7]:
# Retrieve the original lapse dates for eligible renewal records.
# Detect whether dates are stored as Excel serial numbers or text.
# Convert them without changing the original dataset.

date_columns = [
    "date_lapse_insured",
    "date_lapse_policy"
]

df_renewal = df_original.loc[
    (df_original["lapse"] == 2) &
    (df_original["new_business"] == "No")
].copy()

def convert_lapse_dates(series):
    numeric = pd.to_numeric(series, errors="coerce")

    excel_dates = pd.to_datetime(
        numeric,
        unit="D",
        origin="1899-12-30",
        errors="coerce"
    )

    text_dates = pd.to_datetime(
        series.where(numeric.isna()),
        errors="coerce",
        format="mixed",
        dayfirst=True
    )

    return excel_dates.fillna(text_dates)


for col in date_columns:
    df_renewal[col] = convert_lapse_dates(df_renewal[col])

    print(f"\n{col}")
    print("Original non-missing:", df_original.loc[df_renewal.index, col].notna().sum())
    print("Converted non-missing:", df_renewal[col].notna().sum())
    print("Sample dates:", df_renewal[col].dropna().head(5).tolist())


date_lapse_insured
Original non-missing: 15100
Converted non-missing: 15100
Sample dates: [Timestamp('2019-12-31 00:00:00'), Timestamp('2019-01-01 00:00:00'), Timestamp('2019-01-01 00:00:00'), Timestamp('2019-01-01 00:00:00'), Timestamp('2018-12-31 00:00:00')]

date_lapse_policy
Original non-missing: 12608
Converted non-missing: 12608
Sample dates: [Timestamp('2019-12-31 00:00:00'), Timestamp('2019-01-01 00:00:00'), Timestamp('2019-01-01 00:00:00'), Timestamp('2018-12-31 00:00:00'), Timestamp('2018-12-31 00:00:00')]


In [8]:
# Active renewal records should not have termination dates
# in or before their reporting year.
# Missing and future termination dates are not contradictions.

for col in date_columns:

    lapse_year = df_renewal[col].dt.year

    past = (lapse_year < df_renewal["period"]).sum()
    same_year = (lapse_year == df_renewal["period"]).sum()
    future = (lapse_year > df_renewal["period"]).sum()
    missing = lapse_year.isna().sum()

    print(f"\n{col}")
    print("Past:", past)
    print("Same year:", same_year)
    print("Future:", future)
    print("Missing:", missing)


date_lapse_insured
Past: 0
Same year: 0
Future: 15100
Missing: 147556

date_lapse_policy
Past: 0
Same year: 0
Future: 12608
Missing: 150048


In [9]:
# Check whether each converted lapse date matches its
# corresponding recorded lapse year.
# Also count cases where only one of the two fields is missing.

date_year_pairs = {
    "date_lapse_insured": "year_lapse_insured",
    "date_lapse_policy": "year_lapse_policy"
}

for date_col, year_col in date_year_pairs.items():

    actual_year = df_renewal[date_col].dt.year
    recorded_year = pd.to_numeric(
        df_renewal[year_col], errors="coerce"
    )

    mismatch = (
        actual_year.notna() &
        recorded_year.notna() &
        (actual_year != recorded_year)
    )

    missing_disagreement = (
        actual_year.isna() != recorded_year.isna()
    )

    print(f"\n{date_col}")
    print("Year mismatches:", mismatch.sum())
    print("Missing-value disagreements:", missing_disagreement.sum())


date_lapse_insured
Year mismatches: 0
Missing-value disagreements: 0

date_lapse_policy
Year mismatches: 0
Missing-value disagreements: 0


In [10]:
# Flag eligible renewals where either insured-level or
# policy-level termination occurred in or before the reporting year.
# Display any contradictions for manual investigation.

insured_conflict = (
    df_renewal["date_lapse_insured"].dt.year
    <= df_renewal["period"]
)

policy_conflict = (
    df_renewal["date_lapse_policy"].dt.year
    <= df_renewal["period"]
)

conflicts = df_renewal.loc[
    insured_conflict | policy_conflict
]

print("Contradictory renewal records:", len(conflicts))

display(
    conflicts[
        [
            "ID", "ID_policy", "ID_insured",
            "period", "lapse",
            "date_lapse_insured", "date_lapse_policy"
        ]
    ].head(10)
)

Contradictory renewal records: 0


,ID,ID_policy,ID_insured,period,lapse,date_lapse_insured,date_lapse_policy


In [11]:
# Preserve the audited renewal observations.
# Do not automatically remove flagged records.
# Keep lapse dates for later integrity checks.

df_prepared = df_renewal.copy()

print("Final renewal records:", len(df_prepared))
print("\nRecords by year:")
print(df_prepared["period"].value_counts().sort_index())


Final renewal records: 162656

Records by year:
period
2017    55999
2018    53567
2019    53090
Name: count, dtype: int64


## Identifier and Duplicate Verification

In [12]:
# Count unique identifiers in our renewal dataset.
# This helps us understand whether IDs are repeated across records.

id_columns = ["ID", "ID_policy", "ID_insured"]

for col in id_columns:
    print(f"{col}")
    print("Unique values:", df_prepared[col].nunique())
    print("Repeated values:", df_prepared[col].duplicated().sum())
    print("Missing values:", df_prepared[col].isna().sum())
    print()

ID
Unique values: 65161
Repeated values: 97495
Missing values: 0

ID_policy
Unique values: 31392
Repeated values: 131264
Missing values: 0

ID_insured
Unique values: 273
Repeated values: 162383
Missing values: 0



In [13]:
# Examine the relationships between policies and insured individuals.
# Count how many insured IDs belong to each policy.
# Also check whether insured IDs appear under multiple policies.

insured_per_policy = (
    df_prepared.groupby("ID_policy")["ID_insured"]
    .nunique()
)

policies_per_insured = (
    df_prepared.groupby("ID_insured")["ID_policy"]
    .nunique()
)

print("Insured individuals per policy:")
display(insured_per_policy.describe())

print("\nPolicies per insured individual:")
display(policies_per_insured.describe())

Insured individuals per policy:


count    31392.000000
mean         2.075720
std          2.453109
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max        233.000000
Name: ID_insured, dtype: float64


Policies per insured individual:


count      273.000000
mean       238.684982
std       2051.116528
min          1.000000
25%          1.000000
50%          2.000000
75%          6.000000
max      27764.000000
Name: ID_policy, dtype: float64

In [14]:
# Check whether any observations are completely identical.
# Do not remove duplicates yet; first review the results.

exact_duplicates = df_prepared.duplicated()

print("Exact duplicate rows:", exact_duplicates.sum())

display(df_prepared[exact_duplicates].head(10))

Exact duplicate rows: 0


,ID,ID_policy,ID_insured,period,date_effect_insured,date_lapse_insured,date_effect_policy,date_lapse_policy,year_effect_insured,year_lapse_insured,...,C_H,C_GI,C_II,C_IE_P,C_IE_S,C_IE_T,C_GE_P,C_GE_S,C_GE_T,C_C


In [15]:
# Check whether ID is unique within each reporting year.
# Repeated ID-period combinations require investigation.

id_year_duplicates = df_prepared.duplicated(
    subset=["ID", "period"],
    keep=False
)

print("Rows with repeated ID-year combinations:",
      id_year_duplicates.sum())

display(
    df_prepared.loc[
        id_year_duplicates,
        ["ID", "ID_policy", "ID_insured", "period", "premium"]
    ].sort_values(["ID", "period"]).head(20)
)

Rows with repeated ID-year combinations: 0


,ID,ID_policy,ID_insured,period,premium


In [16]:
# Identify policies appearing multiple times within the same year.
# Check whether those records belong to different insured individuals.

policy_year_counts = (
    df_prepared.groupby(["period", "ID_policy"])
    .agg(
        records=("ID", "size"),
        insured_count=("ID_insured", "nunique")
    )
    .reset_index()
)

repeated_policies = policy_year_counts[
    policy_year_counts["records"] > 1
]

print("Policy-year groups with multiple records:",
      len(repeated_policies))

display(repeated_policies.head(10))

Policy-year groups with multiple records: 40217


,period,ID_policy,records,insured_count
2,2017,3,4,4
4,2017,5,2,2
6,2017,8,6,6
8,2017,10,4,4
9,2017,11,2,2
10,2017,12,2,2
12,2017,14,3,3
19,2017,22,7,7
20,2017,23,2,2
21,2017,24,2,2


In [17]:
# Check whether the same insured individual appears more than once
# under the same policy in the same reporting year.
# These combinations may require closer investigation.

contract_columns = ["period", "ID_policy", "ID_insured"]

contract_duplicates = df_prepared.duplicated(
    subset=contract_columns,
    keep=False
)

print("Rows with repeated policy-insured-year combinations:",
      contract_duplicates.sum())

display(
    df_prepared.loc[
        contract_duplicates,
        ["ID", "period", "ID_policy", "ID_insured", "premium"]
    ].sort_values(contract_columns).head(20)
)

Rows with repeated policy-insured-year combinations: 0


,ID,period,ID_policy,ID_insured,premium


In [18]:
# Count how many reporting years each policy appears in.
# Repeated policies across different years are valid observations.
# We should not remove them simply because their IDs repeat.

years_per_policy = (
    df_prepared.groupby("ID_policy")["period"]
    .nunique()
)

print("Number of policies by years observed:")

display(
    years_per_policy.value_counts().sort_index()
)

Number of policies by years observed:


period
1     4972
2     3394
3    23026
Name: count, dtype: int64

In [19]:
# Summarize the identifier and duplicate checks.
# This helps us decide whether any records require further investigation.

duplicate_summary = pd.DataFrame({
    "Check": [
        "Exact duplicate rows",
        "Rows with repeated ID-year",
        "Policy-year groups with multiple records",
        "Rows with repeated policy-insured-year"
    ],
    "Count": [
        int(exact_duplicates.sum()),
        int(id_year_duplicates.sum()),
        len(repeated_policies),
        int(contract_duplicates.sum())
    ]
})

display(duplicate_summary)

,Check,Count
0,Exact duplicate rows,0
1,Rows with repeated ID-year,0
2,Policy-year groups with multiple records,40217
3,Rows with repeated policy-insured-year,0


In [20]:
# Check whether ID uniquely identifies each policy-insured combination.
# Also verify that each policy-insured combination maps to only one ID.
# This will help us understand the actual observation-level identifier.

id_mapping = (
    df_prepared.groupby("ID")[["ID_policy", "ID_insured"]]
    .nunique()
)

contract_mapping = (
    df_prepared.groupby(["ID_policy", "ID_insured"])["ID"]
    .nunique()
)

print("IDs linked to multiple policies:",
      (id_mapping["ID_policy"] > 1).sum())

print("IDs linked to multiple insured numbers:",
      (id_mapping["ID_insured"] > 1).sum())

print("Policy-insured combinations linked to multiple IDs:",
      (contract_mapping > 1).sum())

print("Unique policy-insured combinations:",
      len(contract_mapping))

IDs linked to multiple policies: 0
IDs linked to multiple insured numbers: 0
Policy-insured combinations linked to multiple IDs: 0
Unique policy-insured combinations: 65161


## Date Consistency Checks 

In [21]:
# Read effective dates again from the untouched original dataset.
# Handle both Excel serial numbers and formatted dates such as 6/15/1993.
# Verify that no existing dates are lost during conversion.

effective_date_columns = [
    "date_effect_insured",
    "date_effect_policy"
]

def convert_effective_dates(series):
    numeric = pd.to_numeric(series, errors="coerce")

    excel_dates = pd.to_datetime(
        numeric,
        unit="D",
        origin="1899-12-30",
        errors="coerce"
    )

    text_dates = pd.to_datetime(
        series.where(numeric.isna()),
        format="mixed",
        dayfirst=False,
        errors="coerce"
    )

    return excel_dates.fillna(text_dates)


for col in effective_date_columns:

    original = df_original.loc[df_prepared.index, col]
    converted = convert_effective_dates(original)

    print(f"\n{col}")
    print("Original non-missing:", original.notna().sum())
    print("Converted non-missing:", converted.notna().sum())

    if original.notna().sum() != converted.notna().sum():
        raise ValueError(f"Date conversion needs review: {col}")

    df_prepared[col] = converted

display(df_prepared[["period"] + effective_date_columns].head(10))


date_effect_insured
Original non-missing: 162656
Converted non-missing: 162656

date_effect_policy
Original non-missing: 162656
Converted non-missing: 162656


,period,date_effect_insured,date_effect_policy
1,2017,1993-06-15,1993-06-15
2,2017,1993-06-16,1993-06-16
3,2017,1993-06-15,1993-06-15
4,2017,1993-06-15,1993-06-15
5,2017,1993-06-15,1993-06-15
6,2017,1999-04-19,1993-06-15
7,2017,1995-08-08,1993-07-01
8,2017,1993-07-01,1993-07-01
9,2017,1993-07-01,1993-07-01
12,2017,1993-07-07,1993-07-07


In [22]:
# Compare converted effective dates with their recorded years.
# Check whether each effective date is before, during, or after
# the reporting year.

effective_year_pairs = {
    "date_effect_insured": "year_effect_insured",
    "date_effect_policy": "year_effect_policy"
}

for date_col, year_col in effective_year_pairs.items():

    actual_year = df_prepared[date_col].dt.year
    recorded_year = pd.to_numeric(
        df_prepared[year_col], errors="coerce"
    )
    period = df_prepared["period"]

    mismatch = (
        actual_year.notna()
        & recorded_year.notna()
        & (actual_year != recorded_year)
    )

    print(f"\n{date_col}")
    print("Year mismatches:", mismatch.sum())
    print("Before reporting year:", (actual_year < period).sum())
    print("During reporting year:", (actual_year == period).sum())
    print("After reporting year:", (actual_year > period).sum())
    print("Missing:", actual_year.isna().sum())


date_effect_insured
Year mismatches: 0
Before reporting year: 162656
During reporting year: 0
After reporting year: 0
Missing: 0

date_effect_policy
Year mismatches: 0
Before reporting year: 162656
During reporting year: 0
After reporting year: 0
Missing: 0


In [23]:
# Verify that effective dates do not occur after lapse dates.
# Also compare insured-level and policy-level effective dates.
# Missing lapse dates are excluded from date-order comparisons.

date_pairs = {
    "date_effect_insured": "date_lapse_insured",
    "date_effect_policy": "date_lapse_policy"
}

for effect_col, lapse_col in date_pairs.items():

    both_available = (
        df_prepared[effect_col].notna()
        & df_prepared[lapse_col].notna()
    )

    invalid_order = (
        both_available
        & (df_prepared[effect_col] > df_prepared[lapse_col])
    )

    print(f"\n{effect_col} vs {lapse_col}")
    print("Both dates available:", both_available.sum())
    print("Invalid date order:", invalid_order.sum())


insured = df_prepared["date_effect_insured"]
policy = df_prepared["date_effect_policy"]

print("\nInsured effective before policy:", (insured < policy).sum())
print("Same effective date:", (insured == policy).sum())
print("Insured effective after policy:", (insured > policy).sum())


date_effect_insured vs date_lapse_insured
Both dates available: 15100
Invalid date order: 0

date_effect_policy vs date_lapse_policy
Both dates available: 12608
Invalid date order: 0

Insured effective before policy: 50
Same effective date: 140053
Insured effective after policy: 22553


In [24]:
# Flag genuine date inconsistencies only.
# Avoid treating missing datetime comparisons as year mismatches.
# Do not delete any flagged records automatically.

insured_year = pd.to_numeric(
    df_prepared["year_effect_insured"], errors="coerce"
)

policy_year = pd.to_numeric(
    df_prepared["year_effect_policy"], errors="coerce"
)

date_issues = (
    (
        df_prepared["date_effect_insured"].dt.year.notna()
        & insured_year.notna()
        & (df_prepared["date_effect_insured"].dt.year != insured_year)
    )
    | (
        df_prepared["date_effect_policy"].dt.year.notna()
        & policy_year.notna()
        & (df_prepared["date_effect_policy"].dt.year != policy_year)
    )
    | (
        df_prepared["date_effect_insured"]
        > df_prepared["date_lapse_insured"]
    )
    | (
        df_prepared["date_effect_policy"]
        > df_prepared["date_lapse_policy"]
    )
    | (
        df_prepared["date_effect_insured"].dt.year
        > df_prepared["period"]
    )
    | (
        df_prepared["date_effect_policy"].dt.year
        > df_prepared["period"]
    )
)

print("Records requiring date review:", date_issues.sum())

display(
    df_prepared.loc[
        date_issues,
        [
            "ID", "period",
            "date_effect_insured", "date_effect_policy",
            "date_lapse_insured", "date_lapse_policy"
        ]
    ].head(10)
)

Records requiring date review: 0


,ID,period,date_effect_insured,date_effect_policy,date_lapse_insured,date_lapse_policy


In [25]:
# Compare when the insured individual joined
# with when the corresponding policy became effective.
# Different dates are not necessarily errors.

insured_before_policy = (
    df_prepared["date_effect_insured"]
    < df_prepared["date_effect_policy"]
)

insured_same_day = (
    df_prepared["date_effect_insured"]
    == df_prepared["date_effect_policy"]
)

insured_after_policy = (
    df_prepared["date_effect_insured"]
    > df_prepared["date_effect_policy"]
)

print("Insured effective before policy:",
      insured_before_policy.sum())

print("Both effective on same day:",
      insured_same_day.sum())

print("Insured effective after policy:",
      insured_after_policy.sum())

Insured effective before policy: 50
Both effective on same day: 140053
Insured effective after policy: 22553


In [26]:
# Examine the premium values in our prepared dataset.
# Identify missing, non-numeric, zero, and negative premiums.
# Do not remove any records until we review the results.

premium_numeric = pd.to_numeric(
    df_prepared["premium"],
    errors="coerce"
)

print("Total observations:", len(df_prepared))

print("Missing premium values:",
      df_prepared["premium"].isna().sum())

print("Non-numeric premium values:",
      (
          df_prepared["premium"].notna()
          & premium_numeric.isna()
      ).sum())

print("Zero premiums:",
      (premium_numeric == 0).sum())

print("Negative premiums:",
      (premium_numeric < 0).sum())

print("Positive premiums:",
      (premium_numeric > 0).sum())

print("\nPremium summary:")
display(premium_numeric.describe())

Total observations: 162656
Missing premium values: 0
Non-numeric premium values: 0
Zero premiums: 0
Negative premiums: 0
Positive premiums: 162656

Premium summary:


count    162656.000000
mean        931.374485
std         569.205834
min          33.330000
25%         633.977000
50%         853.740000
75%        1083.275500
max       15339.780000
Name: premium, dtype: float64

In [27]:
# Check premium validity separately for 2017, 2018, and 2019.
# This helps identify whether invalid target values
# are concentrated in a particular reporting year.

premium_check = pd.DataFrame({
    "period": df_prepared["period"],
    "premium": premium_numeric
})

premium_check["missing"] = premium_check["premium"].isna()
premium_check["zero"] = premium_check["premium"] == 0
premium_check["negative"] = premium_check["premium"] < 0
premium_check["positive"] = premium_check["premium"] > 0

summary = premium_check.groupby("period").agg(
    total_records=("premium", "size"),
    missing=("missing", "sum"),
    zero=("zero", "sum"),
    negative=("negative", "sum"),
    positive=("positive", "sum")
)

display(summary)

,total_records,missing,zero,negative,positive
period,,,,,
2017,55999,0,0,0,55999
2018,53567,0,0,0,53567
2019,53090,0,0,0,53090


## Predictor Audit

In [29]:
# Verify that the prepared dataset contains the expected renewal records.
# Confirm that all original columns are preserved.
# Check target validity and duplicate identifiers.
# Do not modify the dataset during verification.

print("Final dataset shape:", df_prepared.shape)

print("\nRecords by reporting year:")
display(df_prepared["period"].value_counts().sort_index())

print("\nOriginal columns:", len(df_original.columns))
print("Prepared columns:", len(df_prepared.columns))

print(
    "Missing original columns:",
    set(df_original.columns) - set(df_prepared.columns)
)

print("\nEligibility violations:", (
    (df_prepared["lapse"] != 2)
    | (df_prepared["new_business"] != "No")
).sum())

print("Invalid premiums:", (
    pd.to_numeric(df_prepared["premium"], errors="coerce")
    .le(0)
    .sum()
))

print("Missing premiums:", df_prepared["premium"].isna().sum())

print("Duplicate ID-year records:",
      df_prepared.duplicated(subset=["ID", "period"]).sum())

print("Duplicate policy-insured-year records:",
      df_prepared.duplicated(
          subset=["ID_policy", "ID_insured", "period"]
      ).sum())

Final dataset shape: (162656, 42)

Records by reporting year:


period
2017    55999
2018    53567
2019    53090
Name: count, dtype: int64


Original columns: 42
Prepared columns: 42
Missing original columns: set()

Eligibility violations: 0
Invalid premiums: 0
Missing premiums: 0
Duplicate ID-year records: 0
Duplicate policy-insured-year records: 0


In [30]:
# Confirm that all 29 candidate predictors are present.
# Keep excluded variables for auditing, but not for model input.
# Identifiers and reporting year will remain available for splitting.

candidate_predictors = feature_groups["Candidate predictor"]

print("Candidate predictors:", len(candidate_predictors))

print(
    "Missing candidate predictors:",
    set(candidate_predictors) - set(df_prepared.columns)
)

print("\nFeature classification:")
display(feature_inventory["Role"].value_counts())

Candidate predictors: 13
Missing candidate predictors: set()

Feature classification:


Role
Needs verification              16
Candidate predictor             13
Exclude from predictors          8
Identifier / split reference     4
Target                           1
Name: count, dtype: int64

In [32]:
# Examine the age column without modifying the dataset.
# Identify missing, non-numeric, negative, and unusually high ages.
# Summarize age values separately for each reporting year.

age = pd.to_numeric(df_prepared["age"], errors="coerce")

print("Total observations:", len(df_prepared))
print("Missing ages:", df_prepared["age"].isna().sum())

print("Non-numeric ages:", (
    df_prepared["age"].notna() & age.isna()
).sum())

print("Negative ages:", (age < 0).sum())
print("Zero ages:", (age == 0).sum())
print("Ages above 100:", (age > 100).sum())
print("Ages above 120:", (age > 120).sum())

print("Fractional ages:", (
    age.notna() & np.isfinite(age) & (age % 1 != 0)
).sum())

print("\nOverall age summary:")
display(age.describe())

print("\nAge summary by reporting year:")
display(
    df_prepared.assign(age_numeric=age)
    .groupby("period")["age_numeric"]
    .agg(["count", "min", "median", "mean", "max"])
)

Total observations: 162656
Missing ages: 0
Non-numeric ages: 0
Negative ages: 0
Zero ages: 528
Ages above 100: 0
Ages above 120: 0
Fractional ages: 0

Overall age summary:


count    162656.000000
mean         48.407117
std          22.778606
min           0.000000
25%          33.000000
50%          50.000000
75%          66.000000
max          99.000000
Name: age, dtype: float64


Age summary by reporting year:


,count,min,median,mean,max
period,,,,,
2017,55999,0,50.0,47.994178,99
2018,53567,0,51.0,48.617134,99
2019,53090,0,51.0,48.630778,99


In [33]:
# Compare ages for the same ID across consecutive available records.
# Calculate the difference in age and reporting year.
# Flag decreases or increases larger than the year gap.
# Do not remove any observations.

age_audit = df_prepared[
    ["ID", "period", "age"]
].copy()

age_audit["age"] = pd.to_numeric(
    age_audit["age"], errors="coerce"
)

age_audit = age_audit.sort_values(
    ["ID", "period"]
)

age_audit["previous_age"] = (
    age_audit.groupby("ID")["age"].shift(1)
)

age_audit["previous_year"] = (
    age_audit.groupby("ID")["period"].shift(1)
)

age_audit["age_difference"] = (
    age_audit["age"] - age_audit["previous_age"]
)

age_audit["year_difference"] = (
    age_audit["period"] - age_audit["previous_year"]
)

age_audit["age_inconsistent"] = (
    (age_audit["age_difference"] < 0)
    | (
        age_audit["age_difference"]
        > age_audit["year_difference"]
    )
)

print("Records with previous-year observations:",
      age_audit["previous_age"].notna().sum())

print("Age inconsistencies:",
      age_audit["age_inconsistent"].sum())

print("\nAge changes:")
display(
    age_audit.groupby(
        ["year_difference", "age_difference"],
        dropna=True
    ).size().reset_index(name="count")
)

print("\nInconsistent records:")
display(
    age_audit.loc[
        age_audit["age_inconsistent"]
    ].head(20)
)

Records with previous-year observations: 97495
Age inconsistencies: 27

Age changes:


,year_difference,age_difference,count
0,1.0,-8.0,1
1,1.0,-3.0,3
2,1.0,-1.0,1
3,1.0,0.0,68
4,1.0,1.0,97399
5,1.0,2.0,8
6,1.0,3.0,2
7,1.0,5.0,3
8,1.0,9.0,2
9,1.0,11.0,1



Inconsistent records:


,ID,period,age,previous_age,previous_year,age_difference,year_difference,age_inconsistent
99976,11371_1,2018,47,45.0,2017.0,2.0,1.0,True
172370,11984_1,2019,60,46.0,2018.0,14.0,1.0,True
172371,11984_2,2019,68,47.0,2018.0,21.0,1.0,True
78666,125_2,2018,68,55.0,2017.0,13.0,1.0,True
154850,1372_1,2019,86,84.0,2018.0,2.0,1.0,True
175890,14137_2,2019,69,67.0,2018.0,2.0,1.0,True
108284,16022_1,2018,74,82.0,2017.0,-8.0,1.0,True
152698,177_2,2019,68,71.0,2018.0,-3.0,1.0,True
82045,1897_2,2018,79,61.0,2017.0,18.0,1.0,True
156166,2229_1,2019,66,64.0,2018.0,2.0,1.0,True


In [34]:
# Identify records with inconsistent age progression.
# Remove only the 27 flagged observations.
# Preserve all other records and verify the final count.

rows_before = len(df_prepared)

inconsistent_indices = age_audit.loc[
    age_audit["age_inconsistent"], 
].index

print("Age-inconsistent records identified:", len(inconsistent_indices))

assert len(inconsistent_indices) == 27, (
    "Expected 27 inconsistent records. Review the age audit."
)

df_prepared = df_prepared.drop(
    index=inconsistent_indices
).copy()

rows_after = len(df_prepared)

print("\nRecords before removal:", rows_before)
print("Records removed:", rows_before - rows_after)
print("Records remaining:", rows_after)

print("\nRecords by reporting year:")
display(df_prepared["period"].value_counts().sort_index())

Age-inconsistent records identified: 27

Records before removal: 162656
Records removed: 27
Records remaining: 162629

Records by reporting year:


period
2017    55999
2018    53555
2019    53075
Name: count, dtype: int64

## DatasetVerification

In [38]:
# Define baseline and extended model predictors.
# Exclude effective dates, identifiers, and leakage-related variables.
# Verify the prepared dataset without removing any additional columns.

baseline_features = [
    "exposure_time",
    "seniority_insured",
    "seniority_policy",
    "type_policy",
    "type_policy_dg",
    "type_product",
    "reimbursement",
    "distribution_channel",
    "gender",
    "age"
]

additional_predictors = [
    "n_insured_pc",
    "n_insured_mun",
    "n_insured_prov",
    "IICIMUN",
    "IICIPROV",
    "C_H",
    "C_GI",
    "C_II",
    "C_IE_P",
    "C_IE_S",
    "C_IE_T",
    "C_GE_P",
    "C_GE_S",
    "C_GE_T",
    "C_C"
]

print("Baseline predictors:", len(baseline_features))
print("Additional predictors:", len(additional_predictors))

print("\nFinal dataset shape:", df_prepared.shape)

print("\nRecords by reporting year:")
display(df_prepared["period"].value_counts().sort_index())

# Verify final data integrity.
premium = pd.to_numeric(df_prepared["premium"], errors="coerce")

assert df_prepared.shape == (162629, 42)
assert set(df_original.columns) == set(df_prepared.columns)

assert (
    (df_prepared["lapse"] == 2) &
    (df_prepared["new_business"] == "No")
).all()

assert np.isfinite(premium).all() and (premium > 0).all()

assert not df_prepared.duplicated(["ID", "period"]).any()

assert not df_prepared.duplicated(
    ["ID_policy", "ID_insured", "period"]
).any()

assert len(baseline_features) == 10
assert len(extended_features) == 25
assert set(extended_features).issubset(df_prepared.columns)

print("\nAll final verification checks passed.")

Baseline predictors: 10
Additional predictors: 15

Final dataset shape: (162629, 42)

Records by reporting year:


period
2017    55999
2018    53555
2019    53075
Name: count, dtype: int64


All final verification checks passed.


In [39]:
# Save the final prepared dataset with all original columns.
# Preserve audit information and both model feature sets.
# Do not overwrite the original insurance dataset.

output_file = "prepared_insurance.csv"

df_prepared.to_csv(output_file, index=False)

print("Dataset saved successfully:", output_file)
print("Rows:", len(df_prepared))
print("Columns:", len(df_prepared.columns))

print("\nBaseline features:", len(baseline_features))
print("Extended features:", len(extended_features))

Dataset saved successfully: prepared_insurance.csv
Rows: 162629
Columns: 42

Baseline features: 10
Extended features: 25
